#### 결측치 확인하기  
결측치는 데이터에서 값이 비어 있는 상태를 말한다.  
Pandas에서는 결측치를 NaN으로 표시하는 경우가 많음. NaN은 숫자 0이나 빈 문자열과 다르다. 값이 없다는 표시이므로 계산이나 조건 처리 전에 먼저 확인해야 한다. 전처리는 값을 바꾸거나 삭제하는 단계부터 시작하지 않는다. 먼저 어느 칼럼에 결측치가 있는지, 몇 개인지, 어떤 행에 포함되는지 파악해야 함.  
그래야 나중에 제거할지, 다른 값으로 채울지 판단할 수 있다.

In [1]:
import pandas as pd

df = pd.read_csv('./data/pandas_preprocess.csv')
print(df.shape)
print(df.head())

(122, 8)
  member_id   age gender signup_date     spend  visit_count   grade  memo
0    MB0001  29.0      F  2025/02/17  204200.0           10   BASIC    정상
1    MB0002  53.0      F  2025/02/17  980000.0            9   BASIC    정상
2    MB0003   NaN      F  2025/09/21   93300.0            9   BASIC  쿠폰사용
3    MB0004  58.0      F  2025/03/09  236900.0            8  SILVER    정상
4    MB0005  24.0      F  2025/08/01   64000.0           10   BASIC   NaN


In [2]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 122 entries, 0 to 121
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   member_id    122 non-null    object 
 1   age          119 non-null    float64
 2   gender       122 non-null    object 
 3   signup_date  122 non-null    object 
 4   spend        118 non-null    float64
 5   visit_count  122 non-null    int64  
 6   grade        122 non-null    object 
 7   memo         104 non-null    object 
dtypes: float64(2), int64(1), object(5)
memory usage: 7.8+ KB
None


In [3]:
print(df.isna().head())

   member_id    age  gender  signup_date  spend  visit_count  grade   memo
0      False  False   False        False  False        False  False  False
1      False  False   False        False  False        False  False  False
2      False   True   False        False  False        False  False  False
3      False  False   False        False  False        False  False  False
4      False  False   False        False  False        False  False   True


In [4]:
print(df.isna().sum())

member_id       0
age             3
gender          0
signup_date     0
spend           4
visit_count     0
grade           0
memo           18
dtype: int64


In [5]:
# 결측치가 있는 컬럼
missing = df.isna().sum()
print(type(missing))
print(missing[missing > 0])

<class 'pandas.core.series.Series'>
age       3
spend     4
memo     18
dtype: int64


In [6]:
# 전체 결측치 개수
print(df.isna().sum().sum())

25


In [7]:
# 결측치가 있는 행 찾기
cond = df.isna().any(axis=1) # 여기서 axis=1은 행 기준으로 결측치가 하나라도 있으면 포함된다는 것!
print(cond.sum())
print(df.loc[cond, ['member_id', 'age', 'spend', 'memo']].head())

25
   member_id   age     spend  memo
2     MB0003   NaN   93300.0  쿠폰사용
4     MB0005  24.0   64000.0   NaN
5     MB0006  47.0       NaN  쿠폰사용
13    MB0014  32.0  254000.0   NaN
27    MB0028  30.0  182900.0   NaN


In [8]:
# 특정 컬럼의 결측치
cond = df['spend'].isna()
print(df.loc[cond, 'member_id'].to_list()) # 'spend' 컬럼이 비어있는 member_id를 출력해서 리스트로 변환

['MB0006', 'MB0057', 'MB0083', 'MB0100']


#### 결측치 확인 시 주의사항  
1. 삭제와 대체 이전 단계  
결측치 확인은 삭제나 대체를 결정하기 전의 단계이다. age가 비어 있따고 해서 무조건 행을 삭제하는 것이 아님.  
분석 목적에 따라 평균이나 중앙값으로 채울 수도 있고, 해당 행을 제외할 수도 있다.  
2. 문자 칼럼의 결측치  
memo처럼 문자형 칼럼에 결측치가 있을 수도 있다. 숫자형 칼럼의 결측치는 평균이나 중앙값으로 대체하는 경우가 있지만, 문자형 칼럼은 '없음'같은 문자열로 채우거나 별도로 분류하는 방식도 가능하다. 이 절에서는 처리하지 않고 확인까지만 다룬다.  
3. 출력 확인  
결측치 개수를 확인한 뒤에는 필요한 컬럼만 선택해 실제 행을 살펴보면 좋다. 숫자만 보고 넘어가면 어떤 값이 비어 있는지 감이 잡히지 않는다.

#### 결측치 제거와 대체  
결측치를 확인했으니 다음 단계는 어떻게 처리할 것인가에 관한 것!  
결측치 처리는 크게 해당 행을 제외하는 방법과 빈 값을 다른 값으로 채우는 방법으로 나뉨  
어느 방법이 항상 정답이라고 할 수는 없으며, 문제의 요구 기준과 분석 목적, 컬럼의 성격에 따라 선택해야 함.  
빅분기 실기에서는 보통 결측치 제거하시오, 평균으로 대체하시오, 특정 값으로 채우시오 같은 처리 조건이 제시됨. 따라서 문제의 지시를 먼저 확인하고 그 조건에 맞는 코드를 정확히 작성하는게 중요!

1. 제거  
제거는 결측치가 있는 행이나 열을 분석 대상에서 제외하는 방식.  
행을 제거하면 계산에 사용하는 데이터 수가 줄어들기에 결측치가 매우 적거나, 해당 값이 없으면 문제 조건을 만족할 수 없는 경우에 사용하기 쉬움!  
2. 대체  
대체는 비어 있는 값을 다른 값으로 채우는 방식이다. 숫자형 컬럼은 평균, 중앙값, 0 등으로 채울 수 있고, 문자형 컬럼은 '기록없음'같은 문자열로 채울 수 있다. 대체를 사용하면 행 수를 유지할 수 있지만, 어떤 값으로 채웠는지에 따라 평균이난 합계가 달라질 수 있다!

#### 행 단위 제거  
1. 전체 결측치 제거  
dropna()는 결측치가 있는 행을 제거한다. 별도 조건을 주지 않는다면 결측치가 하나라도 있는 행은 무조건 제거됨

In [9]:
df_drop_all = df.dropna()

print(df.shape)
print(df_drop_all.shape)

(122, 8)
(97, 8)


2. 특정 컬럼 기준 제거  
특정 컬럼의 결측치만 기준으로 행을 제거할 수 있다.  
subset에 컬럼명을 지정하면 해당 컬럼들에 결측치가 있는 행만 제외함

In [10]:
df_drop_key = df.dropna(subset=['age', 'spend']) # age와 spend 열에서 결측치를 가지는 행만 제거하는 방법
print(df_drop_key.shape)
print(df_drop_key[['member_id', 'age', 'spend', 'memo']].head(6))

(115, 8)
  member_id   age     spend  memo
0    MB0001  29.0  204200.0    정상
1    MB0002  53.0  980000.0    정상
3    MB0004  58.0  236900.0    정상
4    MB0005  24.0   64000.0   NaN
6    MB0007  43.0  129200.0  쿠폰사용
7    MB0008  29.0  297800.0   재방문


#### 값으로 대체  
1. 수치형 칼럼  
fillna()는 결측치를 지정한 값으로 채운다. 숫자형 칼럼에서는 평균이나 중앙값으로 대체하는 경우가 많다.  
다음 코드는 age의 결측치를 age 중앙값으로 채운다.

In [11]:
df_fill_age = df.copy()
median_age = df_fill_age['age'].median()

df_fill_age['age'] = df_fill_age['age'].fillna(median_age)

print(median_age)
print(df_fill_age.loc[df['age'].isna(), ['member_id', 'age']])

37.0
   member_id   age
2     MB0003  37.0
31    MB0032  37.0
77    MB0078  37.0


In [12]:
df_fill_memo = df.copy()
df_fill_memo['memo'] = df_fill_memo['memo'].fillna('기록없음')

print(df_fill_memo.loc[[4, 13, 27], ['member_id', 'memo']])

   member_id  memo
4     MB0005  기록없음
13    MB0014  기록없음
27    MB0028  기록없음


In [13]:
# 금액형 컬럼 - 지출액
# spend처럼 금액을 나타내는 컬럼도 문제 조건에 따라서 0, 평균, 중앙값 등으로 채울 수 있다. 여기서 중앙값으로 대체
df_fill_spend = df.copy()
median_spend = df_fill_spend['spend'].median()
df_fill_spend['spend'] = df_fill_spend['spend'].fillna(median_spend)

print(median_spend)
print(df_fill_spend.loc[df['spend'].isna(), ['member_id', 'spend']])

189450.0
   member_id     spend
5     MB0006  189450.0
56    MB0057  189450.0
82    MB0083  189450.0
99    MB0100  189450.0


In [14]:
# 처리 결과 확인
# 여러 컬럼의 결측치를 한 번에 처리할 때는 원본을 복사한 뒤 컬럼별로 다른 기준을 적용하는 방식이 안전하다.
df_clean = df.copy()

df_clean['age'] = df_clean['age'].fillna(df_clean['age'].median())
df_clean['spend'] = df_clean['spend'].fillna(df_clean['spend'].median())
df_clean['memo'] = df_clean['memo'].fillna('기록없음')

print(df_clean.isna().sum().sum())
print(df_clean.loc[[2,4,5,31], ['member_id', 'age', 'spend', 'memo']])

0
   member_id   age     spend  memo
2     MB0003  37.0   93300.0  쿠폰사용
4     MB0005  24.0   64000.0  기록없음
5     MB0006  47.0  189450.0  쿠폰사용
31    MB0032  37.0  301400.0    정상


#### 원본 보존  
결측치를 제거하거나 대체한 결과를 다시 사용하려면 새로운 변수에 저장하는 것이 좋다. 원본 df를 바로 바꾸면 앞 단계 결과와 비교하기 어려워질 수 있다. 실습 중에는 df_clean처럼 처리 결과를 담는 변수를 따로 두면 흐름을 확인하기 쉽다.

#### 이상치 확인  
이상치는 다른 값들과 비교했을때 유난히 작거나 큰 값을 말한다. 예를 들어 대부분의 지출액이 10만원에서 30만원 사이인데, 한 명만 98만원을 사용했다면 그 값을 먼저 확인해야 할 값들이 된다. 다만 이상치는 무조건 틀린 값이라는 뜻은 아님! 실제로 많이 구매한 일반적인 회원일 수도 있고, 입력 과정에 오류로 인해 0이 하나더 추가된 기록일 수도 있음! 전처리에서는 이상치를 바로 삭제하기 보단 먼저 확인한다. 

1. 극단값  
극단값은 전체 데이터의 일반적인 범위에서 멀리 떨어진 값이다. 평균보다 훨씬 크거나 작을 수 있고,  
최솟값이나 최댓값에서 눈에 띄는 경우가 많다. 작업형 문제에서는 특정 기준보다 큰 값, 특정 기준보다 작은 값, 또는 분위수 기준을 벗어난 값을 찾으라고 제시될 수 있음.  

2. 오류값과 실젯값  
이상치 후보가 항상 오류값은 아니다. 나이가 99세인 값은 매우 크지만 실제로 99세인 사람일수도 있음!  
지출액이 980000인 값도 큰 금액이지만 단체 구매나 고가 구매라면 실젯값일 수 있다. 따라서 이상치 후보를 확인하는 단계와 처리하는 단계는 구분해야 한다.

#### 수치 범위 확인  
1. describe()  
숫자형 컬럼의 범위를 빠르게 볼 떄는 describe()를 사용할 수 있다. count, mean, min, 25%, 50%, 75%, max가 함께 출력되므로 값이 어느 정도 범위에 모여 있는지 확인하기 쉬움! 이때 실습은 앞에서 작업한 회원 정보 데이터프레임 df를 이어서 사용한다. 필요한 칼럼을 선택하고 처리 전후 결과를 비교한다.

In [15]:
print(df[['age', 'spend', 'visit_count']].describe())

              age          spend  visit_count
count  119.000000     118.000000   122.000000
mean    37.537815  189781.355932     9.040984
std     11.745024  116310.494378     3.168537
min     14.000000       0.000000     1.000000
25%     31.000000  115000.000000     7.000000
50%     37.000000  189450.000000     9.000000
75%     43.500000  253800.000000    11.000000
max     99.000000  980000.000000    18.000000


In [16]:
# 정렬 확인
# 최댓값 근처의 행을 직접 보고 싶다면 정렬을 사용하라!
cols = ['member_id', 'age', 'spend', 'visit_count', 'grade']

print(df.sort_values('spend', ascending=False)[cols].head())
print(df.sort_values('age', ascending=False)[cols].head())

    member_id   age     spend  visit_count   grade
1      MB0002  53.0  980000.0            9   BASIC
54     MB0055  32.0  395900.0            7    GOLD
16     MB0017  34.0  344200.0            5   BASIC
102    MB0103  34.0  343900.0           12  SILVER
52     MB0053  54.0  319700.0            9  SILVER
   member_id   age     spend  visit_count   grade
8     MB0009  99.0  109100.0            7  SILVER
71    MB0072  64.0  189000.0            8    GOLD
32    MB0033  58.0  288800.0            4  SILVER
3     MB0004  58.0  236900.0            8  SILVER
23    MB0024  58.0   31800.0            8   BASIC


#### IQR 기준 확인  
IQR은 3분위수에서 1분위수를 뺀 값이다. 판다스에서는 quantile() 함수로 1사분위수와 3사분위수를 구할 수 있음. 일반적으로 1사분위수보다 1.5 X IQR 만큼 작거나 3분위수보다 1.5 X IQR만큼 큰 값을 이상치 후보로 봄

In [17]:
q1 = df['spend'].quantile(0.25)
q3 = df['spend'].quantile(0.75)
iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

print(q1,q3, lower, upper)

cond = ((df['spend'] < lower) | (df['spend'] > upper))
print(df.loc[cond, ['member_id', 'age', 'spend', 'grade']])

115000.0 253800.0 -93200.0 462000.0
  member_id   age     spend  grade
1    MB0002  53.0  980000.0  BASIC


In [18]:
q1 = df['age'].quantile(0.25)
q3 = df['age'].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

print(q1, q3, lower, upper)
cond = ((df['age'] > upper) | (df['age'] < lower))
print(df.loc[cond, ['member_id', 'age', 'spend', 'grade']])

31.0 43.5 12.25 62.25
   member_id   age     spend   grade
8     MB0009  99.0  109100.0  SILVER
71    MB0072  64.0  189000.0    GOLD


#### 이상치 확인 시 주의사항  
1. 문제 조건 우선  
시험 문제에서 'IQR 기준으로 이상치를 찾으시오', '평균에서 표준편차의 2배를 벗어난 값을 찾으시오.'  
'상위 1%를 이상치로 보시오'처럼 기준이 제시되면 그 기준을 우선 따른다. 기준이 다르면 같은 데이터에서도 이상치 개수가 달라질 수 있다.  

2. 삭제 전 확인  
이상치 후보를 찾았다고 해서 바로 삭제하진 않는다. 이상치 확인은 처리의 전 단계이다. 다음 절에서 이상치를 제거하거나 대체하는 방법을 다루지만, 그 전에는 어떤 행이 선택되는지 출력해 확인하는 습관 필요!  

3. 결측치와의 차이  
결측치는 값이 비어 있는 상태이고, 이상치는 값은 존재하지만 다른 값들과 비교해 굉장히 튀는 상태이다.  
따라서 결측치는 isna()로 확인하고, 이상치는 describe(), 정렬, 분위수, 조건식 등을 이용해 확인한다.

#### 4. 이상치 처리  
이상치 처린는 다른 값과 지나치게 떨어진 값을 그대로 둘지, 제거할지, 다른 값으로 바꿀지 결정하는 과정  
데이터에서 계산하는 전처리 기준은 train에서 정하고, 같은 기준을 train과 test에 적용한다.  

처리방향  
1. 제거  
제거는 이상치가 포함된 행을 데이터에서 제외하는 방식  

2. 대체  
대체는 이상치 후보를 다른 값으로 바꾸는 방식

#### 지출액 이상치 처리  
1. 기준 곅산  
먼저 spend 칼럼의 IQR 기준을 계산한다. 1사분위수와 3사분위수를 구하고, 그 차이인 IQR을 이용해 하한값과 상한값을 만든다.

In [19]:
q1 = df['spend'].quantile(0.25)
q3 = df['spend'].quantile(0.75)  
iqr = q3 - q1  

lower = q1 - iqr * 1.5
upper = q3 + iqr * 1.5

print(q1, q3, lower, upper)

115000.0 253800.0 -93200.0 462000.0


In [20]:
# 행 제거  
cond = df['spend'] > upper
print(df.loc[cond, cols])

df_removed = df[(df['spend'].isna()) | (df['spend'] <= upper)] # spend 열이 결측치인 행이 사라지지 않게 함!

print(df_removed.shape)
print(df_removed['spend'].max())

  member_id   age     spend  visit_count  grade
1    MB0002  53.0  980000.0            9  BASIC
(121, 8)
395900.0


In [21]:
# 나이 이상치 처리  
q1 = df['age'].quantile(0.25)
q3 = df['age'].quantile(0.75)
iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

cond = (df['age'] < lower) | (df['age'] > upper)
print(lower, upper)
print(df.loc[cond, cols])

12.25 62.25
   member_id   age     spend  visit_count   grade
8     MB0009  99.0  109100.0            7  SILVER
71    MB0072  64.0  189000.0            8    GOLD


In [22]:
median_age = df['age'].median()

df_age_fixed = df.copy()
df_age_fixed.loc[cond, 'age'] = median_age # age에서 이상치를 가지는 애들을 다 중앙값으로 바꾼다!

print(median_age)
print(df_age_fixed.loc[cond, cols])

37.0
   member_id   age     spend  visit_count   grade
8     MB0009  37.0  109100.0            7  SILVER
71    MB0072  37.0  189000.0            8    GOLD


이상치 처리 시 핵심  
- 원본 보존: 이상치를 처리할 때는 df를 바로 바꾸기보다 df.copy()로 복사본을 만든 뒤 작업하는 것이 안전함. 처리 과정이 잘못되었을 때 원본 데이터로 다시 돌아갈 수 있기 때문이다!  
- 조건식 확인: 제거와 대체는 모든 조건식의 핵심이다. 조건식이 잘못되면 다른 행까지 함께 처리될 수 있다. 따라서 처리하기 전에 df.loc[cond]로 어떤 행이 선택되는지 먼저 출력해보는 습관을 길러라!  
- 문제 조건 우선: 같은 이상치 후보라도 문제에서 요구하는 방식에 따라 답이 달라진다. 삭제하라는 문제에서는 행수가 줄어들고, 상한값으로 대체하라는 문제에서는 행 수가 유지된다. 중앙값으로 대체하라는 조건이 있으면 기준값이 아닌 대푯값으로 바꿔야 한다.

#### 5. 자료형 변환  
자료형 변환으니 컬럼에 저장된 값을 계산, 날짜 처리, 문자열 처리에 맞는 형태로 바꾸는 과정이다.  
CSV 파일을 읽으면 날짜처럼 보이는 값도 문자열로 들어올 수 있고, 결측치가 섞인 숫자 칼럼은 정수가 아니라 실수형으로 읽힐 수 있다.  
작업형 문제에서는 자료형이 맞지 않으면 코드가 실행되지 않거나, 실행되더라도 원하는 결과가 나오지 않을 수 있다. 따라서 데이터를 불러온 뒤에는 먼저 자료형을 확인하고, 필요한 컬럼만 변환한다.

In [23]:
# 자료형 확인  
cols = ['member_id', 'age', 'signup_date', 'spend', 'visit_count', 'grade']
print(df[cols].head())
print(df[cols].dtypes)

  member_id   age signup_date     spend  visit_count   grade
0    MB0001  29.0  2025/02/17  204200.0           10   BASIC
1    MB0002  53.0  2025/02/17  980000.0            9   BASIC
2    MB0003   NaN  2025/09/21   93300.0            9   BASIC
3    MB0004  58.0  2025/03/09  236900.0            8  SILVER
4    MB0005  24.0  2025/08/01   64000.0           10   BASIC
member_id       object
age            float64
signup_date     object
spend          float64
visit_count      int64
grade           object
dtype: object


날짜형으로 바꾸면 아래에서 dt.month처럼 연도, 월, 일 같은 정보를 dt 접근자로 쉽게 꺼낼 수 있다

In [24]:
# 날짜형 변환
df_date = df.copy()
df_date['signup_date'] = pd.to_datetime(df_date['signup_date'])

print(df_date['signup_date'].dtypes)
print(df_date[['member_id', 'signup_date']].head())
print(df_date['signup_date'].dt.month.head().to_list())

datetime64[ns]
  member_id signup_date
0    MB0001  2025-02-17
1    MB0002  2025-02-17
2    MB0003  2025-09-21
3    MB0004  2025-03-09
4    MB0005  2025-08-01
[2, 2, 9, 3, 8]


In [25]:
median_age = df['age'].median()

df_age = df.copy()
df_age['age_filled'] = df_age['age'].fillna(median_age).astype(int)

print(median_age)
print(df_age[['member_id', 'age', 'age_filled']].head(6))
print(df_age['age_filled'].dtype)

37.0
  member_id   age  age_filled
0    MB0001  29.0          29
1    MB0002  53.0          53
2    MB0003   NaN          37
3    MB0004  58.0          58
4    MB0005  24.0          24
5    MB0006  47.0          47
int32


In [26]:
# 금액 칼럼을 변환
df_spend = df.copy()
df_spend['spend_filled'] = df_spend['spend'].fillna(0).astype(int)

print(df_spend[['member_id', 'spend', 'spend_filled']].head(7))
print(df_spend['spend_filled'].dtype)

  member_id     spend  spend_filled
0    MB0001  204200.0        204200
1    MB0002  980000.0        980000
2    MB0003   93300.0         93300
3    MB0004  236900.0        236900
4    MB0005   64000.0         64000
5    MB0006       NaN             0
6    MB0007  129200.0        129200
int32


#### 범주형 변환  
1. 표기 통일  
범주형 컬럼은 자료형만 바꾸기 전에 값의 표기가 섞여 있는지 확인해야한다. grade 컬럼에는 BASIC과 basic처럼 대소문자가 섞인 값이 있으므로, 먼저 대문자로 통일한다.

In [27]:
df_grade = df.copy()
df_grade['grade'] = df_grade['grade'].str.upper()

grade_counts = df_grade['grade'].value_counts().sort_index()
print(grade_counts.rename_axis(None).to_string())

BASIC     46
GOLD      22
SILVER    49
VIP        5


문자 범주를 category 자료형으로 바꾸면 해당 컬럼을 범주형 데이터로 명확하게 다룰 수 있다.  
시험 풀이에서 반드시 category로 바꾸어야 하는 경우는 많지 않지만, 범주형 데이터의 종류 확인시 도움

In [28]:
# category 자료형
df_grade['grade_cat'] = df_grade['grade'].astype('category')

print(df_grade['grade_cat'].dtype)
print(df_grade['grade_cat'].cat.categories.to_list())

category
['BASIC', 'GOLD', 'SILVER', 'VIP']


변환 함수의 종류  

|날짜 문자열 변환 | pd.to_datetime() |  
|---|---|  
|숫자형 변환 | fillna() 후 astype(int) |  
|범주형 변환 | str.upper() 후 astype('category')|  

특히 결측치가 있는 컬럼은 astype(int)를 적용하면 오류가 날 수 있음. 먼저 격측치를 어떻게 처리할지 정하고  
그 다음에 자료형을 바꾸는 순서로 접근한다.

#### 6. 파생변수 생성  
파생변수는 기존 컬럼을 이용해 새로 만든 컬럼이다. 주문 수량과 단가를 곱해 주문 금액을 만들거나, 주문 날짜에서 월을 꺼내 월별 분석에 사용할 수 있다. 기존 데이터 직접적으로 포함된 건 아니지만 기존 값을 조합해서 필요한 새 칼럼을 만드는 과정!  
작업형 1에서는 파생변수를 만든 뒤 조건 필터링, 정렬, 집계를 이어서 수행하는 문제가 자주 등장한다.  
작업형 2에서도 모델 학습에 사용할 특징을 만들 때 파생변수를 사용할 수 있다. 이 절에서는 복잡한 함수를 쓰기보다 산술 계산, 날짜 추출, 조건 결과 저장처럼 시험에서 바로 쓰기 쉬운 형태를 먼저 익힌다.

In [29]:
orders = pd.read_csv('./data/pandas_orders.csv')
cols = ['order_id', 'qty', 'unit_price', 'discount_rate', 'order_date', 'sales_amount']

print(orders[cols].head())

  order_id  qty  unit_price  discount_rate  order_date  sales_amount
0   O00001   11       29100           0.05  2026-04-10        304095
1   O00002   11       91700           0.00  2026-02-22       1008700
2   O00003    2       60300           0.03  2026-03-11        116982
3   O00004    5       60500           0.00  2026-04-24        302500
4   O00005    5       10000           0.00  2026-04-23         50000


#### 계산 컬럼 생성  
1. 수량과 단가의 곱  
가장 기본적인 파생변수는 컬럼끼리 계산해서 만든다. 수량과 단가를 곱해서 할인 전 주문 금액을 만들 수 있다.  

In [30]:
df_calc = orders.copy()
df_calc['gross_amount'] = df_calc['qty'] * df_calc['unit_price']

print(df_calc[['order_id', 'qty', 'unit_price', 'gross_amount', 'sales_amount']].head())

  order_id  qty  unit_price  gross_amount  sales_amount
0   O00001   11       29100        320100        304095
1   O00002   11       91700       1008700       1008700
2   O00003    2       60300        120600        116982
3   O00004    5       60500        302500        302500
4   O00005    5       10000         50000         50000


In [31]:
df_calc['discount_amount'] = (df_calc['gross_amount'] * df_calc['discount_rate']).round().astype(int)
df_calc['pay_amount'] = df_calc['gross_amount'] - df_calc['discount_amount']

print(df_calc[['order_id', 'gross_amount', 'discount_rate', 'discount_amount', 'pay_amount', 'sales_amount']].head())

  order_id  gross_amount  discount_rate  discount_amount  pay_amount  \
0   O00001        320100           0.05            16005      304095   
1   O00002       1008700           0.00                0     1008700   
2   O00003        120600           0.03             3618      116982   
3   O00004        302500           0.00                0      302500   
4   O00005         50000           0.00                0       50000   

   sales_amount  
0        304095  
1       1008700  
2        116982  
3        302500  
4         50000  


#### 날짜 파생변수 생성  
날짜 컬럼에서 연도, 월, 일을 꺼내 새 컬럼으로 만들 수도 있다. order_date는 CSV를 읽은 직후에는 문자열이므로, 먼저 날짜형으로 바꾼 뒤 월을 추출한다.

In [32]:
df_date = orders.copy()
df_date['order_date'] = pd.to_datetime(df_date['order_date'])
df_date['order_month'] = df_date['order_date'].dt.month

print(df_date[['order_id', 'order_date', 'order_month']].head())
print(df_date['order_month'].value_counts())

  order_id order_date  order_month
0   O00001 2026-04-10            4
1   O00002 2026-02-22            2
2   O00003 2026-03-11            3
3   O00004 2026-04-24            4
4   O00005 2026-04-23            4
order_month
4    73
3    69
2    56
5     2
Name: count, dtype: int64


#### 조건 결과 저장  
조건식의 결과도 새 칼럼으로 저장할 수 있다. 예를 들어 매출액이 500000원 이상인 주문을 큰 주문으로 표시하면,  
이후 조건 필터링이나 집계에 다시 사용할 수 있다

In [33]:
df_flag = orders.copy()
df_flag['large_order'] = df_flag['sales_amount'] >= 500000

print(df_flag[['order_id', 'sales_amount', 'large_order']].head(8))
print(df_flag['large_order'].value_counts())

  order_id  sales_amount  large_order
0   O00001        304095        False
1   O00002       1008700         True
2   O00003        116982        False
3   O00004        302500        False
4   O00005         50000        False
5   O00006        197400        False
6   O00007         48060        False
7   O00008         53900        False
large_order
False    137
True      63
Name: count, dtype: int64


파생변수 이름 붙이기  
파생변수 이름은 계산 의미가 드러나게 붙이는 것이 좋다. gross_amount는 할인 전 금액, discount_amount는 할인액, order_month는 주문 월처럼 읽는 사람이 바로 이해할 수 있는 이름이 안전하다. a, b, temp 같은 이름은 시험 중에는 빠를 수 있지만, 코드가 길어지면 실수하기 쉽다!

#### 값 변경  
데이터 전처리에서는 값을 새로 계산하는 것뿐만 아니라 이미 들어 있는 값을 일정한 기준에 맞게 고치는 일도 자주 수행한다. 같은 성별인데 F와 f가 섞여 있거나, 같은 등급인데 BASIC과 basic이 함께 있으면 집계 결과가 나뉘어 나온다. 이때에는 값을 통일한 뒤 분석해야 한다.  
값 변경은 원본 데이터를 무조건 덮어쓰는 작업이 아님. 먼저 어떤 값이 섞여 있는지 확인하고, 바꿀 규칙을 정한 뒤, 변경 결과를 다시 확인하는 흐름으로 진행한다.

In [34]:
# 변경 대상 확인  
print(df[['member_id', 'gender', 'grade', 'memo', 'age', 'spend']].head(12))
print(df['gender'].value_counts(dropna=False))
print(df['grade'].value_counts(dropna=False))

   member_id gender   grade  memo   age     spend
0     MB0001      F   BASIC    정상  29.0  204200.0
1     MB0002      F   BASIC    정상  53.0  980000.0
2     MB0003      F   BASIC  쿠폰사용   NaN   93300.0
3     MB0004      F  SILVER    정상  58.0  236900.0
4     MB0005      F   BASIC   NaN  24.0   64000.0
5     MB0006      f  SILVER  쿠폰사용  47.0       NaN
6     MB0007      F   BASIC  쿠폰사용  43.0  129200.0
7     MB0008      M    GOLD   재방문  29.0  297800.0
8     MB0009      m  SILVER   재방문  99.0  109100.0
9     MB0010      F   BASIC   재방문  33.0  195000.0
10    MB0011      F   basic   재방문  31.0  316700.0
11    MB0012      F    GOLD   재방문  55.0  299600.0
gender
F          49
M          46
f          12
m           8
unknown     7
Name: count, dtype: int64
grade
SILVER    41
BASIC     36
GOLD      22
basic     10
silver     8
VIP        5
Name: count, dtype: int64


먼저 세고나서 나중에 바꾸기  
값을 바꾸기 전에 value_counts()로 현재 값의 종류를 먼저 확인하는 습관이 필요하다.  
어떤 값이 들어 있는지 모른 채 replace()부터 사용하면 일부 값만 바뀌고 나머지는 그대로 남을 수 있다.

1. replace를 이용한 변경  
replace()는 특정 값을 다른 값으로 바꿀 때 사용한다. 성별 컬럼에서 f는 F로, m은 M으로 바꾸고, unknown은 U로 바꾸어 표기를 통일한다.

In [35]:
df_fix = df.copy()
df_fix['gender'] = df_fix['gender'].replace({'f':'F', 'm':'M', 'unknown':'U'})
print(df_fix['gender'].value_counts(dropna=False))

gender
F    61
M    54
U     7
Name: count, dtype: int64


2. 문자열 메서드를 이용한 대문자 통일  
문자열 값 전체를 같은 규칙으로 바꿀 때는 str.upper()나 str.lower()를 사용할 수 있다.  
등급 컬럼은 일부 값만 소문자로 되어 있으므로 전체를 대문자로 통일한다.

In [36]:
df_fix['grade'] = df_fix['grade'].str.upper()
print(df_fix['grade'].value_counts(dropna=False))

grade
SILVER    49
BASIC     46
GOLD      22
VIP        5
Name: count, dtype: int64


#### 조건에 맞는 값 수정  
조건에 맞는 행만 골라 특정 칼럼의 값을 바꿀 때는 loc을 사용한다. 예를 들어, 이용 금액이 500000원 이상인 회원의 등급을 VIP로 바꿀 수 있다

In [37]:
cond = df_fix['spend'] >= 500000
print(df_fix.loc[cond, ['member_id', 'spend', 'grade']].head())
df_fix.loc[cond, 'grade'] = 'VIP'

print(df_fix.loc[cond, ['member_id', 'spend', 'grade']].head())
print(df_fix['grade'].value_counts())

  member_id     spend  grade
1    MB0002  980000.0  BASIC
  member_id     spend grade
1    MB0002  980000.0   VIP
grade
SILVER    49
BASIC     45
GOLD      22
VIP        6
Name: count, dtype: int64


#### 조건 필터링과 값 변경을 함께 사용하는 경우 loc[행 조건, 바꿀 컬럼] 구조를 기억하면 된다!

In [38]:
# 비어 있는 표싯값 입력
# 결측치를 분석에서 제외할 수도 있지만 , 문자 컬럼에서는 빈 값을 하나의 표싯값으로 바꿔 관리하기도 함
# memo 칼럼의 빈 값을 기록없음으로 바꾸면 이후 value_counts()에서 별도 항목으로 확인할 수 있다.

print(df_fix['memo'].isna().sum())
df_fix.loc[df_fix['memo'].isna(), 'memo'] = '기록없음'

print(df_fix[['member_id', 'memo']].head(8))
print('결측치 개수 :',df_fix['memo'].isna().sum())

18
  member_id  memo
0    MB0001    정상
1    MB0002    정상
2    MB0003  쿠폰사용
3    MB0004    정상
4    MB0005  기록없음
5    MB0006  쿠폰사용
6    MB0007  쿠폰사용
7    MB0008   재방문
결측치 개수 : 0


#### 매핑 테이블 활용  
등급명 변환: 값마다 대응되는 설명을 붙이고 싶을 때는 map()을 사용할 수 있다. 등급 코드를 한글 설명으로 바꿔 새 컬럼에 저장하면 결과를 읽기 쉬워진다.

In [39]:
grade_map = {'BASIC': '일반', 'SILVER' : '우수', 'GOLD' : '골드', 'VIP' : '최우수'}
df_fix['grade_label'] = df_fix['grade'].map(grade_map) # 코드값을 설명값으로 바꾸거나, 범주값을 다른 이름으로 바꿀 때 유용하다

print(df_fix[['member_id', 'grade', 'grade_label']].head(10))
print(df_fix['grade_label'].value_counts(dropna=False))

  member_id   grade grade_label
0    MB0001   BASIC          일반
1    MB0002     VIP         최우수
2    MB0003   BASIC          일반
3    MB0004  SILVER          우수
4    MB0005   BASIC          일반
5    MB0006  SILVER          우수
6    MB0007   BASIC          일반
7    MB0008    GOLD          골드
8    MB0009  SILVER          우수
9    MB0010   BASIC          일반
grade_label
우수     49
일반     45
골드     22
최우수     6
Name: count, dtype: int64


#### 값 변경 방법 비교  
값을 바꾼 뒤에는 head()로 일부 행만 보는 것보다 value_counts()나 isna().sum()으로 전체 결과를 확인하는 것이 좋다!  
일부 행은 제대로 바뀌어 보여도, 다른 행에 남아 있는 값이 있을 수 있기 때문이다.  
|함수|용도|예시|  
|--|--|--|  
|replace()|특정 값으로 바꿀 때|f->F, unkown->U|  
|str.upper()|문자열 표기를 한 번에 통일할 때|basic->BASIC|  
|loc['조건, 컬럼']|조건에 맞는 행의 값을 바꿀 때|고액 이용자의 등급 변경|  
|map()|값과 설명을 대응시킬 때|BASIC->일반|  


#### 중복 데이터 처리  
데이터를 여러 번 수집하거나 파일을 합치는 과정에서는 같은 행이 반복해서 들어갈 수 있다. 같은 회원이 두 번 들어 있거나, 같은 거래가 중복으로 저장되면, 개수와 합계가 실제보다 크게 계산이 된다. 중복 데이터 처리는 분석 전에 데이터가 한 번만 반영되도록 정리하는 과정이다.  
중복 데이터는 단순히 눈으로 찾기 어려움!! 행이 많아지면 같은 행이 어디에 있는지 확인하기 힘들기 때문에 duplicated() 함수로 중복 여부를 먼저 확인하고, 필요한 경우 drop_duplicated()로 제거한다

In [40]:
# 중복 여부 확인  
print(df.shape)
print(df.duplicated().sum())
print(df.duplicated(keep=False).sum()) # keep=False인 경우 원본처럼 남겨둘 첫 번째 행까지 모두 True로 표시한다.

(122, 8)
2
4


In [41]:
dup = df.duplicated(keep=False)

print(df.loc[dup, ['member_id', 'age', 'gender', 'signup_date', 'spend', 'visit_count', 'grade', 'memo']])

    member_id   age gender signup_date     spend  visit_count  grade memo
10     MB0011  31.0      F  2025/11/22  316700.0            9  basic  재방문
25     MB0026  50.0      M  2026/05/07  126100.0            7   GOLD   정상
120    MB0011  31.0      F  2025/11/22  316700.0            9  basic  재방문
121    MB0026  50.0      M  2026/05/07  126100.0            7   GOLD   정상


In [42]:
# 중복 행 제거  
# drop_duplicates()는 중복 행 제거에 사용함. 기본 설정에서는 처음 나온 행은 남기고, 뒤에 나온 중복행만 제거함
df_drop = df.drop_duplicates()

print(df_drop.shape)
print(df_drop.duplicated().sum())

(120, 8)
0


In [43]:
# 인덱스 재정리
# 중복 행을 제거해도 기존 인덱스 번호는 그대로 남을 수 있다. 행 번호를 0부터 다시 정리하고 싶으면 reset_index(drop=True)를 이어서 사용한다
df_clean = df.drop_duplicates().reset_index(drop=True)

print(df_clean.shape)
print(df_clean.tail(3))

(120, 8)
    member_id   age gender signup_date     spend  visit_count   grade memo
117    MB0118  43.0      f  2025/06/08   85300.0           11  SILVER  재방문
118    MB0119  39.0      m  2026/03/31  234100.0           10  SILVER   정상
119    MB0120  19.0      F  2025/07/04  252200.0            8   basic   정상


#### 기준 컬럼 지정  
모든 컬럼이 같을 때만 중복으로 볼 수도 있지만, 특정 칼럼을 기준으로 중복을 판단해야 하는 경우도 있음.  
예를 들어, 회원 ID는 한 회원을 구분하는 값이므로 member_id가 반복되면 같은 회원이 중복 입력되었는지 확인해야한다.

In [44]:
print(df.duplicated(subset=['member_id']).sum())
cond = df.duplicated(subset=['member_id'], keep=False)

print(df.loc[cond, ['member_id', 'age', 'spend', 'grade']])

2
    member_id   age     spend  grade
10     MB0011  31.0  316700.0  basic
25     MB0026  50.0  126100.0   GOLD
120    MB0011  31.0  316700.0  basic
121    MB0026  50.0  126100.0   GOLD


실제 데이터에서는 ID는 같은데 일부 정보가 다른 경우도 있음. 그런 경우에는 어떤 행을 남길지 문제 조건을 먼저 확인해야함

In [45]:
print(df.duplicated(subset=['member_id'], keep='first').sum()) # 처음 나온 행을 남기는 것
print(df.duplicated(subset=['member_id'], keep='last').sum()) # 마지막 행은 남긴다
print(df.duplicated(subset=['member_id'], keep=False).sum()) # 중복 묶음 전체 표시

2
2
4


#### 9.문자열 처리  
문자열 처리는 문자로 저장된 데이터를 필요한 형태로 정리하는 과정이다.  
전화번호에서 하이픈을 제거하거나, 이메일에서 도메인만 분리하거나, 서비스 코드에서 앞부분만 추출하는 작업이 여기에 해당한다.  
빅분기 실기에서는 문자열을 복잡하게 다루기보다 특정 문자를 포함하는 행을 찾거나, 앞뒤 일부 문자를 잘라 계산 조건으로 쓰는 경우가 많다. 따라서 문자열 처리에서는 .str 접근자를 이용해서 컬럼 단위로 문자를 처리하는 방법을 먼저 익혀야 한다.  

In [46]:
text_df = pd.read_csv('./data/pandas_dates_strings.csv')

cols = ['reservation_id', 'user_code', 'phone', 'email', 'service_code', 'status']
print(text_df[cols].head(6).to_string(index=False))

reservation_id user_code         phone                  email service_code    status
        R00001    U2516E 010-3749-6274 user001@example1.co.kr       BK-041  reserved
        R00002    U1133A 010-7802-5340 user002@example2.co.kr       GY-008 cancelled
        R00003    U9501E 010-4965-9180 user003@example3.co.kr       GY-017  reserved
        R00004    U2273C 010-3509-4010 user004@example4.co.kr       BK-005 cancelled
        R00005    U6897E 010-6025-8075 user005@example0.co.kr       CL-030   visited
        R00006    U3959D 010-9056-5734 user006@example1.co.kr       CL-037   visited


#### 문자열 접근자  
판다스에서 컬럼 안의 문자열을 처리할 때는 컬럼 뒤에 .str을 붙인다. 파이썬 문자열 하나를 처리하는 것이 아니라, 데이터프레임의 한 컬럼에 들어 있는 여러 문자열을 한꺼번에 처리하기 때문이다.

In [47]:
print(text_df['service_code'].str[:2].head().to_string(index=False))

BK
GY
GY
BK
CL


문자열 하나를 다룰 때는 'ABC[:2]'처럼 인덱싱 할 수 있지만 데이터프레임의 컬럼은 문자열 하나가 아니라 여러 문자열이 세로로 모여 있는 시리즈입니다. 그래서 판다스 df['컬럼'].str처럼 문자열 전용 통로를 거친 뒤 자르기, 바꾸기, 포함 여부 확인을 하도록 합니다.

In [52]:
text_df['phone_clean'] = text_df['phone'].str.replace('-', '', regex=False)
text_df['phone_mid'] = text_df['phone'].str.split('-').str[1]

print(text_df[['phone', 'phone_clean', 'phone_mid']].head(8).to_string(index=False))

        phone phone_clean phone_mid
010-3749-6274 01037496274      3749
010-7802-5340 01078025340      7802
010-4965-9180 01049659180      4965
010-3509-4010 01035094010      3509
010-6025-8075 01060258075      6025
010-9056-5734 01090565734      9056
010-6459-5063 01064595063      6459
          NaN         NaN       NaN


코드값 분리  
서비스 코드처럼 문자와 숫자가 함께 있는 값은 앞부분과 뒷부분을 나누어 사용할 수 있다. 앞의 두 글자는 서비스 그룹으로, 뒤의 세 자리는 서비스 번호로 분리해보자!

In [49]:
# 코드값 분리 
text_df['service_group'] = text_df['service_code'].str[:2]
text_df['service_no'] = text_df['service_code'].str[-3:].astype(int)

print(text_df[['service_code', 'service_group', 'service_no']].head(8).to_string(index=False))
print(text_df)

service_code service_group  service_no
      BK-041            BK          41
      GY-008            GY           8
      GY-017            GY          17
      BK-005            BK           5
      CL-030            CL          30
      CL-037            CL          37
      ST-039            ST          39
      GY-014            GY          14
    reservation_id user_code          phone                   email  \
0           R00001    U2516E  010-3749-6274  user001@example1.co.kr   
1           R00002    U1133A  010-7802-5340  user002@example2.co.kr   
2           R00003    U9501E  010-4965-9180  user003@example3.co.kr   
3           R00004    U2273C  010-3509-4010  user004@example4.co.kr   
4           R00005    U6897E  010-6025-8075  user005@example0.co.kr   
..             ...       ...            ...                     ...   
145         R00146    U6189C  010-7993-7158  user146@example1.co.kr   
146         R00147    U7631E  010-3899-4378  user147@example2.co.kr   
147       

#### 코드값 분리  
서비스 코드처럼 문자와 숫자가 함께 있는 값은 앞부분과 뒷부분을 나누어 사용할 수 있다. 앞의 두 글자는 서비스 그룹으로,  
뒤의 세 자리는 서비스 번호로 분리해보자!  

In [51]:
text_df['service_group'] = text_df['service_code'].str[:2]
text_df['service_no'] = text_df['service_code'].str[-3:].astype(int)

print(text_df[['service_code', 'service_group', 'service_no']].head(8).to_string(index=False))
print(text_df['service_group'].value_counts().sort_index())

service_code service_group  service_no
      BK-041            BK          41
      GY-008            GY           8
      GY-017            GY          17
      BK-005            BK           5
      CL-030            CL          30
      CL-037            CL          37
      ST-039            ST          39
      GY-014            GY          14
service_group
BK    37
CL    26
GY    33
PT    25
ST    29
Name: count, dtype: int64


#### 문자열 조건  
1. 포함 여부 확인  
문자열 조건은 특정 글자를 포함하는 행만 골라낼 때 사용한다. 이메일 주소에서 특정 도메인을 포함하는 행을 찾거나, 서비스 코드가 특정 문자로 시작하는 행을 찾을때 주로 사용함

In [53]:
cond = text_df['email'].str.contains('example1.co.kr', regex=False, na=False)

print(cond.sum())
print(text_df.loc[cond, ['reservation_id', 'email', 'status']].head().to_string(index=False))

30
reservation_id                  email   status
        R00001 user001@example1.co.kr reserved
        R00006 user006@example1.co.kr  visited
        R00011 user011@example1.co.kr  visited
        R00016 user016@example1.co.kr reserved
        R00021 user021@example1.co.kr  no_show


str.contains()는 조건에 맞는지 True 혹은 False로 반환한다. na=False는 결측치가 있을 때 False로 처리하겠다는 의미

In [58]:
# 시작 문자 확인
cond = text_df['service_code'].str.startswith('BK', na=False)

print(cond.sum())
print(text_df.loc[cond, ['reservation_id', 'service_code', 'status']].head().to_string(index=False))

37
reservation_id service_code    status
        R00001       BK-041  reserved
        R00004       BK-005 cancelled
        R00009       BK-028   visited
        R00014       BK-014  reserved
        R00016       BK-020  reserved


str.startswith('BK')는 문자열이 'BK'로 시작하는지 확인한다. 서비스 코드의 앞부분이 곧 서비스 그룹을 뜻하는 데이터라면,  
슬라이싱으로 service_group을 만든 뒤 조건을 걸 수도 있고, startswith()로 바로 조건을 만들 수도 있다

#### 문자열 길이와 도메인
1. 전화번호 길이 확인
문자열 길이는 str.len()로 확인한다. 전화번호에서 하이픈을 제거한 뒤 길이를 확인하면 형식이 맞는 전화번호가 몇 개인지 빠르게 확인할 수 있다. 국내 휴대전화 번호는 하이픈을 빼면 11자리이므로, 길이가 11이 아닌 값은 형식이 어긋난 후보로 볼 수 있다.

In [59]:
print(text_df['phone_clean'].str.len().value_counts(dropna=False).sort_index().to_string())
print((text_df['phone_clean'].str.len() == 11).sum())

phone_clean
11.0    148
NaN       2
148


2. 이메일 도메인 분리  
이메일 주소는 항상 사용자명@도메인 구조이므로 @를 기준으로 나누면 원하는 부분만 뽑아낼 수 있다.
str.split('@')은 각 값을 리스트로 쪼개고 뒤에 붙는 .str[1]이 리스트의 두 번째 요소인 도메인만 가져온다.

In [60]:
# 이메일 도메인 분리  
text_df['domain'] = text_df['email'].str.split('@').str[1]

print(text_df[['email', 'domain']].head().to_string(index=False))
print(text_df['domain'].value_counts().sort_index().to_string())

                 email         domain
user001@example1.co.kr example1.co.kr
user002@example2.co.kr example2.co.kr
user003@example3.co.kr example3.co.kr
user004@example4.co.kr example4.co.kr
user005@example0.co.kr example0.co.kr
domain
example0.co.kr    30
example1.co.kr    30
example2.co.kr    30
example3.co.kr    30
example4.co.kr    30


.str.split('@').str[1]하면 @ 뒤쪽 값만 가져올 수 있음  
str[0] -> 앞에 아이디 부분

문장열 처리는 그 자체가 끝이 아니라 조건 필터링이나 집계로 이어지는 경우가 많다. 예를 들어 서비스 코드 앞 두 글자를 잘라 service_group을 만들고, 그다음 특정 그룹의 예약 건수를 세는 식이다. 문자열을 자른 뒤에는 그 결과를 새 컬럼으로 저장해두면 다음 계산에서 반복 코드를 줄일 수 있다.

In [62]:
import pandas as pd

df = pd.read_csv('./data/pandas_dates_strings.csv')

print(df['reserved_at'].dtype)

df['reserved_at'] = pd.to_datetime(df['reserved_at'])

print(df['reserved_at'].dtype)
print(df[['reservation_id', 'reserved_at']].head(3))

object
datetime64[ns]
  reservation_id         reserved_at
0         R00001 2026-05-12 08:00:00
1         R00002 2026-04-13 20:00:00
2         R00003 2026-03-24 10:00:00


In [63]:
df['reserve_month'] = df['reserved_at'].dt.month
df['reserve_day'] = df['reserved_at'].dt.day
df['reserve_hour'] = df['reserved_at'].dt.hour

print(df[['reservation_id', 'reserve_month', 'reserve_day', 'reserve_hour']].head())

  reservation_id  reserve_month  reserve_day  reserve_hour
0         R00001              5           12             8
1         R00002              4           13            20
2         R00003              3           24            10
3         R00004              4           21            10
4         R00005              3            4            13


In [65]:
# 특정 조건에 맞는 것 찾기
cond = df['reserved_at'] >= '2026-05-01'

print(cond.sum())
print(df[cond][['reservation_id', 'reserved_at', 'status']].head())

64
   reservation_id         reserved_at     status
0          R00001 2026-05-12 08:00:00   reserved
9          R00010 2026-05-07 09:00:00  cancelled
12         R00013 2026-05-05 16:00:00    visited
14         R00015 2026-05-13 20:00:00   reserved
15         R00016 2026-05-19 16:00:00   reserved


#### 날짜 데이터 처리  
CSV 파일에 들어 있는 날짜는 화면에서는 날짜처럼 보이지만, 판다스로 불러오면 문자 자료형으로 인식되는 경우가 많다. 날짜가 문자로 저장된 상태에서는 연도, 월, 일, 시간 같은 값을 바로 꺼내기 어렵다. 따라서 날짜 계산이나 날짜 기준 필터링을 하려면 먼저 날짜형으로 변환해야 한다.  
판다스에서는 pd.to_datetime()을 사용해 문자 형태의 날짜를 날짜 자료형으로 바꾼다.  
날짜형으로 변환한 뒤에는 .dt 접근자를 사용해 연도, 월, 일, 요일, 시간 등을 추출할 수 있다.  
작업형 1에서는 특정 월의 데이터만 고르거나, 예약일에서 월, 일을 뽑아 계산하는 방식으로 자주 응용된다.

In [74]:
df = pd.read_csv('./data/pandas_preprocess.csv')

df_copy = df.copy()
df_copy.info()

q3 = df_copy['spend'].quantile(0.75)
q1 = df_copy['spend'].quantile(0.25)
IQR = q3 - q1

upper = q3 + 1.5 * IQR
lower = q1 - 1.5 * IQR
cond = df_copy['spend'] > upper
df_copy.loc[cond, ['spend']] = upper

print(df_copy.loc[cond, ['member_id', 'age', 'spend']])

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 122 entries, 0 to 121
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   member_id    122 non-null    object 
 1   age          119 non-null    float64
 2   gender       122 non-null    object 
 3   signup_date  122 non-null    object 
 4   spend        118 non-null    float64
 5   visit_count  122 non-null    int64  
 6   grade        122 non-null    object 
 7   memo         104 non-null    object 
dtypes: float64(2), int64(1), object(5)
memory usage: 7.8+ KB
  member_id   age     spend
1    MB0002  53.0  462000.0


In [84]:
df = pd.read_csv('./data/pandas_preprocess.csv')
df_copy = df.copy()

df_copy['signup_date'] = pd.to_datetime(df_copy['signup_date'])

df_copy['signup_year'] = df_copy['signup_date'].dt.year

print((df_copy['signup_year'] == 2026).sum())


28
